## Data Generator for Optimization Parameters

This code preprocesses parameters for input to the Alternatives for Resilient Communities Tornado Model

The inputs include geospatial data related to neighborhoods, distribution poles and lines, buildings, and tornado path or event definition

The outputs include the number of buildings, casualty rates, and, for the specified event, the building damage states, dislocation probabilities,
debris amounts, and power system fragilities

Some of the data are preprocessed further for simplification to speed up optimization runtimes (e.g., by summing across all damage states here
instead of inside the optimization pipeline)

In [1]:
import copy
import geopandas as gp
import math
import numpy as np
import numpy.ma as ma
import pandas as pd
import pickle
import re
from scipy.stats import norm
from scipy.stats import lognorm
import os
import warnings
pd.set_option('mode.chained_assignment', None)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
warnings.filterwarnings('ignore')

# SET UP

In [2]:
# ====================== Define scenarios: if 1, use probability, if 0 use actual storm 

event = 'Joplin' # define event, e.g., 'Joplin', 'EF0', 'Weighted'

run_census = True

In [12]:
def load_data():

    '''Function to load data (building dataset, neighborhoods, tornado path, occurence rates, etc.)'''
    
    prj =  "EPSG:26998" # Projection system for Joplin

    # ============= Infrastructure data ===================
    
    buildings = gp.read_file(rf"Buildings/Bldg_inventory.shp").to_crs(prj).reset_index(drop=True) # Building portfoilio
    
    # =============== Neighborhoods ========================
    
    neighborhoods = gp.read_file(rf"Neighborhoods/neighborhoods_original.shp").to_crs(prj).reset_index(drop=True)
    
    neighborhoods = neighborhoods[neighborhoods['geometry'].isna()==False].reset_index(drop=True)
    
    # =============== Historic tornado path ================
    
    tornado = gp.read_file(rf"Tornado/tornado.shp").to_crs(prj).reset_index(drop=True) # Load historic 2011 tornado path/intensity

    event_lambda = np.array(pd.read_excel("community_data.xlsx","Weighted_Events",index_col=0))[0] # occurence rates of different EF storms for weighted case

    # ============== Fragility parameters =========================

    mu_matrix = pd.read_excel("community_data.xlsx","Fragility_Mu",index_col=0) # mu
    sigma_matrix = pd.read_excel("community_data.xlsx","Fragility_Sigma",index_col=0) # sigma
    
    # Indices of different archetypes
    indices = pd.read_excel("community_data.xlsx","Indices") # indices of different buildings
    
    archetype = indices['Code'] # archetype corresponding to each index

    retro_class = indices['Retro_class'] # get retrofit class 0-5

    EF_wind = np.array([29,38,50,61,75,89]) # vector of wind speeds corresponding to EF0, EF1, etc.

    return buildings, neighborhoods, tornado, event_lambda, mu_matrix, sigma_matrix, archetype, retro_class, indices, EF_wind

def map_archetype_retrofit(param): 
    
    '''Function to map archetypes to corresponding retrofit cases
    Input: param: a vector of values for each archetype (e.g., a 1-d vector for all 17 arcetypes)
    Output: vect: a vector of values for each archetype/retrofit combo (e.g., a 1-d vector for all 57 archetype/retrofit combos'''
    
    codes = indices['Code'][0:param.shape[0]]
    vect = np.zeros(mu_matrix.shape[0])
    j = 0
    
    for i in codes: # Loop through all archetype codes and assign building costs
        vect[indices['Code']==i] = param[j]
        j+=1
        
    return vect


def collect_census_data(neighborhoods):

    '''Function to merge census data with neighborhood shapefile'''
    
    ## ========== Load census data ======================
    data_household = pd.read_csv(rf"Census_data/household_data.csv").drop(0).reset_index() # Household info
    data_race = pd.read_csv(rf"Census_data/race_data.csv").drop(0).reset_index() # Race info
    data_workers = pd.read_csv(rf"Census_data/worker_data.csv").drop(0).reset_index() # Worker info

    # =========== Reduce and rename columns =============
    data_household = data_household[['NAME','S1101_C01_002E','S1101_C01_019E']].rename(columns={'S1101_C01_002E':'AvgHHsize','S1101_C01_019E':'PercRenter'})
    data_race = data_race[['NAME','DP05_0001E','DP05_0067E','DP05_0068E','DP05_0073E']].rename(columns={'DP05_0001E':'TotPop','DP05_0067E':'PercAA','DP05_0068E':'PercNA','DP05_0073E':'PercHisp'})
    data_workers = data_workers[['NAME','S2404_C01_006E']].rename(columns= {'S2404_C01_006E':'Industrial'})

    # ========== Reformat, calculate percentages, and join the 3 datasets =========
    data_census = data_household.merge(data_race,on='NAME')
    data_census['PercRenter'] = data_census['PercRenter'].astype(float)
    data_census['AvgHHsize'] = data_census['AvgHHsize'].astype(float)
    data_census['TotPop'] = data_census['TotPop'].astype(float)
    data_census['PercAA'] = data_census['PercAA'].astype(float)/data_census['TotPop']*100
    data_census['PercNA'] = data_census['PercNA'].astype(float)/data_census['TotPop']*100
    data_census['PercHisp'] = data_census['PercHisp'].astype(float)/data_census['TotPop']*100
    data_census = data_census.merge(data_workers,on='NAME')
    data_census['IndustPerc'] = data_census['Industrial'].astype(int)/data_census['TotPop']*100

    # ============ Join census data to neighborhood data ==============

    for i in np.arange(0,data_household.shape[0]):
        
        census_name = data_household.iloc[i:i+1,0].str.split(';')[i][0].split(' ')[2]
        if '.' in census_name:
            census_name = re.sub('[.]', '', census_name)
            census_name = '0'+census_name
        else:
            census_name = '0'+census_name+'00'
        data_census['NAME'][i] = census_name
    
    neighborhoods = neighborhoods.merge(data_census,right_on='NAME', left_on= 'TRACTCE20') 

    return neighborhoods

def get_building_portfolio(neighborhoods):

    '''Function to collect building portfolio (number of buildings of each type in each neighborhood)'''

    # ===============================================
    # Map defined archetypes and merge with neighborhood shapefile
    # ==================================================

    # The archetypes here are defined as per and therefore need to be mapped to the system used in this study
    
    archetype_dict = {1:0,5:1,6:2,7:3,8:4,9:5,10:6,11:7,12:8,13:9,14:10,15:11,16:12,17:13,18:14,19:15}
    buildings['archetype'] = buildings['archetype'].replace(archetype_dict)

    # ===============================
    # ======= Get number of buildings
    # ===============================

    neighborhoods['NumBldgs']=0  # Initialize total building count
    
    for j in np.array(indices['Code'].unique()): # loop through different building archetypes

        neighborhoods['NumBldgs'+str(j)]=0 # Initialize 
    
        # Join buildings based on neighborhood 
        pointsInPolygon = gp.sjoin(buildings[buildings['archetype']==j], neighborhoods, how="inner", predicate='intersects')
    
        # Loop through all neighborhoods and count buildings of that type
        
        for i in np.arange(neighborhoods.shape[0]):
            neighborhoods['NumBldgs'+str(j)][i] = pointsInPolygon[pointsInPolygon['GEOID20']==neighborhoods['GEOID20'][i]].shape[0]
            neighborhoods['NumBldgs'][i] += pointsInPolygon[pointsInPolygon['GEOID20']==neighborhoods['GEOID20'][i]].shape[0]

    
    return neighborhoods

def get_wind_speeds(neighborhoods):

    '''Function to get wind speeds for each neighborhood under the defined event (e.g., Joplin)'''
    
    # Let's consider only neighborhoods with buildings (drop any without any buildings)

    neighborhoods = neighborhoods[neighborhoods['NumBldgs']>0]


    # If we are defining specified EF event, we can set all neighborhoods to the corresponding wind speed
    
   
       
    wind_ef_dict = {'EF0':29,'EF1':38,'EF2': 50, 'EF3':61, 'EF4':75,'EF5':89} # Wang 2022
    tornado['wind'] = tornado['ef_rating']
    tornado['wind'] = tornado['wind'].replace(wind_ef_dict)

    # =========== Join buildings to corresponding EF intensity ========================

    # Join to nearest tornado and drop duplicates
    neighborhoods = neighborhoods.sjoin_nearest(tornado,how='left',distance_col='dist').drop(columns='index_right').reset_index(drop=True)
    neighborhoods = neighborhoods.sort_values('wind', ascending=False).drop_duplicates(['geometry']).reset_index(drop=True)

    # Drop winds with no tornado intensity and replace nan
    neighborhoods['wind'] = neighborhoods['wind'].replace(np.nan,0)
    neighborhoods['wind'][neighborhoods['dist']>0]=0

    if event[0:2] == 'EF':
        neighborhoods['wind'] = EF_wind[int(event[2])]

    return neighborhoods

def get_x_b_0():

    ''' Function to get matrix of intial building configuration'''
    
    # Initial building configuration

    x_b_0 = np.zeros([neighborhoods.shape[0],mu_matrix.shape[0]]) # initialize
    
    j = 0 # initialize
    
    uniqueindices = indices['ID'][(indices['Sheltered']==0)&(indices['Retrofit']==0)] # get unique indices of 19 base archetypes
    
    for i in np.arange(uniqueindices.shape[0]):
        x_b_0[:,i] = neighborhoods['NumBldgs'+str(indices['Code'][i])]

    return x_b_0

def p_ds(wind,mu_vector,sigma_vector):

    ''' Function to calculate probability of building being in given damage state 0 to 4
    Inputs: 
    wind: wind speed (m/s)
    mu_vector: vector of mu coefficients for DS0-4
    sigma_vector: vector of sigma coefficients for DS0-4
    Ouput: 
    DS_prob: vector of probability of being in given damage state 0 to 4'''
    
    DS_prob = np.zeros(5) # initialize 

    # For DS 1 to 3, set a max value of 0. This deals with rare cases where the fragility models have e.g., DS2 as more probable than DS1
    DS_prob[1] = np.maximum(lognorm.cdf(wind,scale=math.exp(mu_vector[0]),s=sigma_vector[0])-lognorm.cdf(wind,scale=math.exp(mu_vector[1]),s=sigma_vector[1]),0)
    DS_prob[2] = np.maximum(lognorm.cdf(wind,scale=math.exp(mu_vector[1]),s=sigma_vector[1])-lognorm.cdf(wind,scale=math.exp(mu_vector[2]),s=sigma_vector[2]),0)
    DS_prob[3] = np.maximum(lognorm.cdf(wind,scale=math.exp(mu_vector[2]),s=sigma_vector[2])-lognorm.cdf(wind,scale=math.exp(mu_vector[3]),s=sigma_vector[3]),0)
    DS_prob[4] = lognorm.cdf(wind,scale=math.exp(mu_vector[3]),s=sigma_vector[3])
    DS_prob[0] = 1-np.sum(DS_prob[1:])
    return DS_prob

def get_DS_prob_matrix(event,event_lambda):

    '''Function to obtain damage statte probability matrix for various archetypes in a given neighborhood
    Event: string of event name, e.g., 'EF5'
    event_lambda: Frequency of event'''
    
    # Initialize as matrix of neighborhoods x mu for DS0-4 x sigma for DS0-4
    DS_prob_matrix = np.zeros([neighborhoods.shape[0],mu_matrix.shape[0],5])
    
    # =========== If weighted case, weight damages across all cases
    
    if event == 'Weighted': 
        
        for j in np.arange(mu_matrix.shape[0]): # Loop through all archetypes
            
            for k in np.arange(EF_wind.shape[0]): # Loop through all wind intensities
                
                DS_prob_matrix[:,j,:] += event_lambda[k]*p_ds(EF_wind[k],mu_matrix.iloc[j,:],sigma_matrix.iloc[j,:]) # calculate damage state probability
    
    else: # For all other cases, calculate direct damages
        
        for i in np.arange(neighborhoods.shape[0]): # Loop through all neighborhoods
            
            for j in np.arange(mu_matrix.shape[0]): # Loop through all archetypes
                
                DS_prob_matrix[i,j,:] = p_ds(neighborhoods['wind'][i],mu_matrix.iloc[j,:],sigma_matrix.iloc[j,:]) # calculate damage state probability
    
    # Calculate DS0
    
    DS_prob_matrix[:,:,0] = 1-np.sum(DS_prob_matrix[:,:,1:],axis=2)
    
    # Create matrix for calculating casualties, where the DS of a sheltered home is taken as 0 at 100%, and all other damage states have 0 probabiltiy
    
    sheltered_indices = np.where(indices['Sheltered']==1) # get sheltered indices
    DS_prob_matrix_for_casualties = copy.deepcopy(DS_prob_matrix) # create sheltered DS probability 
    
    DS_prob_matrix_for_casualties[:,sheltered_indices,0]=1  # set DS0 to 100%
    DS_prob_matrix_for_casualties[:,sheltered_indices,1:]=0 # set all other Damage States to 0%

    return DS_prob_matrix, DS_prob_matrix_for_casualties

def get_building_costs():

    '''Function to get costs of buildings under various damage states'''
    
    building_costs = np.array(pd.read_excel("community_data.xlsx","Building_Costs",index_col=0)['Cost']) # Load data

    # Get costs of repairs vector
    repair_prop_vector = [0,0.02,0.1,0.5,1] # proportion of damages for each damage state
    
    # Initialize repair costs 
    repair_cost_vector = np.zeros([5,building_costs.shape[0]]) # initialize cost of repairing building
    
    for i in np.arange(5): # Loop through all damage states and calculate cost to repair
        repair_cost_vector[i,:] = repair_prop_vector[i]*building_costs

    return repair_cost_vector

def percent_displaced(DS,p_AA,p_NA,p_renter):

    '''Function to calculate probability of displacement given DS and socioeconomic conditions
    Inputs:
    DS: damage state 0 to 4
    p_AA, p_NA, p_renter: proportion of African American/ Native American/ renter in neighborhood
    Output:
    pd: probability of displacement'''

    # Dislocation parameters from Lumberton study by van de Lindt

    a1 = -1.46 # Constant 
    a2 = [0,2.2,3.88,3.88,3.888] # coefficient for damage state
    a3 = 0.98 # coefficient for African American proportion
    a4 = 1.78 # coefficient for Native American proportion
    a5 = 1.06 # coefficient for renter proportion

    pd = 1/(1+math.exp(-(a1 + a2[DS] + a3*p_AA +a4*p_NA + a5*p_renter))) # percent displaced

    return pd

def get_displacements(neighborhoods):

    '''Obtain displacement probabilities for each neighborhood for different damage states'''

    prob_disp = np.zeros([neighborhoods.shape[0],5]) # initialize probability of displacement for each neighborhood and damage stae

    for i in np.arange(neighborhoods.shape[0]): # loop through all neighborhoods
        for DS in np.arange(5): # Loop through all damage states
    
            # calculate dispalcement probability 
            prob_disp[i,DS] = percent_displaced(DS,neighborhoods['PercAA'][i]/100,neighborhoods['PercNA'][i]/100,neighborhoods['PercRenter'][i]/100)
            
    return prob_disp

def get_repair_times():

    '''Function to extract matrix of restoration times for varying archetypes and damage states'''
    
    # Get repair time mu and sigma (Koliou et al.) assuming building is functional when it reaches 75% of its original capacity
    
    mu_restoration = pd.read_excel("community_data.xlsx","Repair_time_mu",index_col=0) # restoration time mu for given DS and archetype
    sigma_restoration = pd.read_excel("community_data.xlsx","Repair_time_sigma",index_col=0) # restoration time sigma for given DS and archetype
    
    # Expected restoration time, mean of lognormal distribution
    
    t_rest = np.exp(mu_restoration+0.5*sigma_restoration**2) # days to reach functionality 75%
    t_rest['DS1'] = 0 # if DS1, it is immediately habitable
    t_rest= np.hstack((np.zeros([t_rest.shape[0],1]),np.array(t_rest))) # resize

    # Map expected restoration times to retrofit/archetype combinations
    
    expected_rest_time = np.zeros([mu_matrix.shape[0],t_rest.shape[1]])
    for i in np.arange(t_rest.shape[1]):
        expected_rest_time[:,i] = map_archetype_retrofit(t_rest[:,i]) # matrix of expected repair times for archetype/retrofit and DS state

    # Compute expected repair time for each neighborhood
    rest_time_expected = np.empty([neighborhoods.shape[0],mu_matrix.shape[0],5])
    for i in np.arange(mu_matrix.shape[0]):
        for j in np.arange(5):
            rest_time_expected[:,i,j] =DS_prob_matrix[:,i,j]*expected_rest_time[i,j] # Expected restoration times weighted by their probabilities

    
    return rest_time_expected 

def get_repair_costs(rest_time_expected):

    '''Function to get repair cost matrix of various building archetypes and damage levels'''
    
    # Get floor area and map to archetypes
    
    crent = 0.89*(3.28**2) # rent disruption, 1/m2

    cd = 13.89 # disruption cost, /m2/month 
    fa = np.array(pd.read_excel("community_data.xlsx","FloorAreas",index_col=0)) # floor areas (m2) for archetypes
 
    fa_vector = map_archetype_retrofit(fa) # Floor area vector 


    # Calculate expected restoration costs per m2 (divide restorationt time by 30 to get from days to month)
    rest_cost_expected = (np.array(neighborhoods['PercRenter']/100*cd)[:,np.newaxis,np.newaxis]
                          +np.array(1-neighborhoods['PercRenter']/100)[:,np.newaxis,np.newaxis]*(cd+crent*rest_time_expected/30))
    
    # Update to 0 if no restoration time
    rest_cost_expected[rest_time_expected==0]=0 

    # Calculate expected restoration costs based on floor area and dislocation probability
    
    for i in np.arange(mu_matrix.shape[0]): # Loop through all archetypes
        
        rest_cost_expected[:,i,:] = rest_cost_expected[:,i,:]*fa_vector[i]*prob_disp
        
    rest_cost_expected = rest_cost_expected.sum(axis=2) # sum across all damage states
    
    # Restrict costs to residential only
    rest_cost_expected[:,indices['Class']!='Res']=0 # expected restoration costs for a given archetype in a given neighborhood

    return rest_cost_expected, fa_vector

def get_casualty_rates():
    
    '''Casualties from HAZUS-MH earthquake model assuming 5 pm hit, W1 residential structure
    50% are assumed to be at home at 5 pm, assuming no residents will be outside at this time
    Apply a correction for severe and fatal injuries to match historic 2011 Joplin tornado'''

    calibration_factor = 7.25 # adjustment factor for severe and extreme casualties- this is calcualted specific to Joplin MO

    casualty_rates = np.zeros([4,4],dtype=object) # initialize 
    
    casualty_rates[:3,:] = np.array([[0.05, 0, 0, 0],
                      [0.25, 0.03, 0, 0],
                      [1, 0.1, 0.001, 0.001]])/100 # Define injury rate given Damage State (vertical) and injury level (horizontal)
    
    casualty_rates[3,:] = (0.97*np.asarray([5, 1, 0.01, 0.01])+0.03*np.asarray([40,20,3,5]))/100 # Account for complete collapse
    
    casualty_rates[:,2:] = casualty_rates[:,2:]*calibration_factor # adjust for tornado underestimation

    return casualty_rates

def distribute_population():

    '''Function to distribute the population to different building archetypes'''

    # Set up some data
    
    retro_class = indices['Retro_class'] # get retrofit class 0-5
    indices_noretrofit = indices[(indices['Retrofit']==0)&(indices['Sheltered']==0)] # get nonretrofit indices
    total_buildings = x_b_0[:,(indices['Retrofit']==0)&(indices['Sheltered']==0)].sum(axis=0) # get number of buildings
    total_population = np.matmul(neighborhoods['AvgHHsize'],x_b_0[:,indices['Class']=='Res']).sum() # get population
    fa_noret = fa_vector[(indices['Retrofit']==0)&(indices['Sheltered']==0)] # get floor area of each structure

    # ============= Estimate number of people at each location =======================

    '''Define distributions of residents here (residential, industrial, commercial)
    For 5:00 p.m. model, assume half of residents are in residential homes, and the remaining residents are evenly split at industrial/commercial based on their 
    floor areas'''

    perc_res = 0.5 # proportion of people in residential buildings 

    perc_comm = 0.1

    perc_work = 0.4

    perc_industrial = neighborhoods['IndustPerc'].mean()/100*perc_work # proportion of people in industrial buildings
    perc_commercial = perc_res - perc_industrial # percentage of residents in commercial building at time of tornado
    
    # get breakdown of residents
    num_residents = np.zeros(indices_noretrofit.shape[0]) #initialize

    # Get distribution of people in commercial buildings, distributed based on the floor area
    num_residents[indices_noretrofit['Commercial']==1] = perc_commercial*(total_buildings[indices_noretrofit['Commercial']==1]*fa_noret[indices_noretrofit['Commercial']==1]*total_population)/np.sum((total_buildings[indices_noretrofit['Commercial']==1]*fa_noret[indices_noretrofit['Commercial']==1]))
    num_residents[indices_noretrofit['Class']=='Ind'] = perc_industrial*(total_buildings[indices_noretrofit['Class']=='Ind']*fa_noret[indices_noretrofit['Class']=='Ind']*total_population)/np.sum((total_buildings[indices_noretrofit['Class']=='Ind']*fa_noret[indices_noretrofit['Class']=='Ind']))

    # Write population distribution to excel file

    building_proportions = pd.DataFrame(columns=['ID']) # intialize
    building_proportions['ID'] = indices_noretrofit['ID']
    building_proportions['Name'] = indices_noretrofit['Name']
    building_proportions['Buildings'] = total_buildings # get total number of each archetype
    building_proportions['Building_proportion'] = building_proportions['Buildings']/total_buildings.sum() # get proportion of each archetype
    building_proportions['People'] = num_residents # get number of residents
    building_proportions['People_proportion'] = num_residents/total_population # get proportion of residents
    building_proportions['People_per_building'] = num_residents/total_buildings # get proportion of residents 

    # write results
    with pd.ExcelWriter("community_data.xlsx", engine="openpyxl", mode="a",if_sheet_exists="replace") as writer:
        pd.DataFrame(building_proportions).to_excel(writer, sheet_name="building_proportions")


    
    # Load number of residents in a given archetype 
    building_proportions = np.array(pd.read_excel("community_data.xlsx","building_proportions")['People_per_building'])

    # get final residents per building
    residents_per_building = np.array(np.repeat(building_proportions,neighborhoods.shape[0])).reshape([building_proportions.shape[0],neighborhoods.shape[0]])
    
    # set residential based on neighborhood average household size (assuming only half are in home at time)
    residents_per_building[indices['Class'][0:archetype.max()+1]=='Res',:] = neighborhoods['AvgHHsize']*perc_res

    return residents_per_building

def get_debris():

    '''Function to get debris amount generated by different archetypes of different damage levels'''
    #HAZUS MH Technical Manual Chapter 12
    
    debris_size = np.zeros([mu_matrix.shape[0],5,neighborhoods.shape[0]]) # initialize, archetypes x damage states x neighborhoods

    debris_proportions = np.array([0,0,0.05,0.34,1]) #proportion of debris given damage state 0-5

    building_areas = np.array(fa_vector) # building area, m3, based on the proportion of open area (debris esimtating guide FEMA, assume 3 m height but 1/3 is open)
    
    for i in np.arange(building_areas.shape[0]):
        for k in np.arange(debris_proportions.shape[0]):
            debris_size[i,k,:] = debris_proportions[k]*building_areas[i]*DS_prob_matrix[:,i,k]
    debris_size = debris_size.sum(axis=1).T # debris volume for given scenario (m3)

    return debris_size

# Reformulate building portfolio

def reformat_2D_matrix(inp_vect):

    # Reformat a 2D matrix to 3d (to consider both archetype and retrofit status)
    
    output_vect = np.zeros([inp_vect.shape[0],archetype.max()+1,retro_class.max()+1]) # initialize output vector
    
    for i in np.arange(x_b_0.shape[0]):
        for j in np.arange(archetype.max()+1):
            for k in np.arange(retro_class.max()+1):
                loc = np.where((retro_class==k)&(archetype==j))
                if len(loc[0]) > 0:
                    output_vect[i,j,k] = inp_vect[i,loc] # update to 3D
    return output_vect 

def reformat_DS_matrix(inp_vect):
    
    # Reformulate damage state matrix to 4D (neighborhoods, archetype, retrofit, and DS)
    
    output_vect =  np.zeros([neighborhoods.shape[0],archetype.max()+1,retro_class.max()+1,5]) 
    for i in np.arange(inp_vect.shape[0]):
        for j in np.arange(archetype.max()+1):
            for k in np.arange(retro_class.max()+1):
                for l in np.arange(inp_vect.shape[2]):
                    loc = np.where((retro_class==k)&(archetype==j))
                    if len(loc[0]) > 0:
                        
                         output_vect[i,j,k,l] = inp_vect[i,loc,l] # update to 4D

    return output_vect



def write_results():
    
    # Write building debris volume for different neighborhoods/archetypes

    debris_size_updated = reformat_2D_matrix(debris_size)
    
    output = open('building_debris'+event+'.pkl', 'wb')
    pickle.dump(debris_size_updated, output)
    output.close()
    
    # Write initial building portfolio 

    x_b_0_updated = reformat_2D_matrix(x_b_0)
    
    output = open('x_b_0.pkl','wb')
    pickle.dump(x_b_0_updated, output)
    output.close()
    
    # Write expected restoration costs 
    rest_cost_expected_updated = reformat_2D_matrix(rest_cost_expected)
    
    output = open('restoration_costs'+event+'.pkl','wb')
    pickle.dump(rest_cost_expected_updated,output)
    output.close()
    
    # Write residents per archetype

    output = open('residents_per_bld.pkl', 'wb')
    pickle.dump(residents_per_building, output)
    output.close()
    
    # Write casualty rates

    output = open('casualty_rates.pkl', 'wb')
    pickle.dump(casualty_rates, output)
    output.close()
    
    # Write Damage state probabilities

    DS_prob_matrix_updated = reformat_DS_matrix(DS_prob_matrix)

    output = open('DS_prob_matrix'+event+'.pkl', 'wb')
    pickle.dump(DS_prob_matrix_updated, output)
    output.close()
    
    # Considering shelter possibility

    DS_prob_matrix_for_casualties_updated =  reformat_DS_matrix(DS_prob_matrix_for_casualties)

    output = open('DS_prob_matrix_for_casualties'+event+'.pkl', 'wb')
    pickle.dump(DS_prob_matrix_for_casualties_updated, output)
    output.close()

In [15]:

for event in ['EF0','EF1','EF2','EF3','EF4','EF5']: # define event, e.g., 'Joplin', 'EF0', 'Weighted'

    '''Run this to generate the necessary data files to run ARC-T for a given event
    event: event name, e.g., 'EF5'
    run_census: Set to true if the neighborhood file already needs to be linked to census data to collect the '''
    
    # Load data
    buildings, neighborhoods, tornado, event_lambda, mu_matrix, sigma_matrix, archetype, retro_class, indices, EF_wind = load_data()
    
    # Get census data if needed
    if run_census == True:        
        neighborhoods = collect_census_data(neighborhoods)
    
    # Get building portfolio
    neighborhoods= get_building_portfolio(neighborhoods)
    
    # Get wind speed corresponding to event
    neighborhoods = get_wind_speeds(neighborhoods)
    
    # Get initial building vector
    x_b_0 = get_x_b_0()
    
    # Get costs of repairing buildings
    repair_cost_vector = get_building_costs()
    
    # Get damage state probability matrices
    DS_prob_matrix,DS_prob_matrix_for_casualties=get_DS_prob_matrix(event,event_lambda)
    
    # Get displacement probability
    prob_disp = get_displacements(neighborhoods)
    
    # Get restoration times
    rest_time_expected = get_repair_times()
    
    # Get repair costs
    rest_cost_expected, fa_vector = get_repair_costs(rest_time_expected)
    
    # Get casualty rates
    casualty_rates = get_casualty_rates()
    
    # Get building sizes
    residents_per_building = distribute_population()
    
    # Get debris size
    
    debris_size = get_debris()
    
    # Write results
    
    write_results()